**Step 1: Library installation**

In [1]:
!pip install -U langgraph langchain langchain-google-genai langchaincommunity tavily-python requests
print("--- Library installation complete. ---")
print("Please RESTART YOUR SESSION NOW: Runtime -> Restart session")


ERROR: Could not find a version that satisfies the requirement langchaincommunity (from versions: none)
ERROR: No matching distribution found for langchaincommunity
--- Library installation complete. ---
Please RESTART YOUR SESSION NOW: Runtime -> Restart session


In [2]:
!pip install --upgrade langchain-google-genai
print("langchain-google-genai package installed. Please restart runtime and try again.")


  Using cached langchain_google_genai-4.2.1-py3-none-any.whl.metadata (2.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.3 MB/s eta 0:00:00
langchain-google-genai package installed. Please restart runtime and try again.


In [3]:
!pip install -U tavily-python requests
print("--- Library installation complete. ---")
print("Please RESTART YOUR SESSION NOW: Runtime -> Restart session")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
--- Library installation complete. ---
Please RESTART YOUR SESSION NOW: Runtime -> Restart session


**Step 2: API Keys**

In [1]:
from google.colab import userdata
import os

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["OMDB_API_KEY"] = userdata.get("OMDB_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Movie-Agentic-RAG"

print("--- API Keys and Environment Setup Complete ---")
keystocheck = ["GEMINI_API_KEY", "LANGSMITH_API_KEY", "OMDB_API_KEY", "TAVILY_API_KEY"]
missingkeys = [key for key in keystocheck if not os.environ.get(key)]
if missingkeys:
    print(f"WARNING: Missing keys: {missingkeys}. Add them to Colab Secrets and re-run.")
else:
    print("All API keys loaded successfully.")


--- API Keys and Environment Setup Complete ---
All API keys loaded successfully.


In [2]:
import subprocess
import sys

print("📦 Installing RAG dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                      "chromadb", "langchain-community"])
print("✅ ChromaDB and langchain-community installed successfully!")


📦 Installing RAG dependencies...
✅ ChromaDB and langchain-community installed successfully!


**Step 3: Agent Toolkit**

In [3]:
#Define Enhanced Agent Tools (Quota-Friendly Version)
import requests
import json
from langchain.tools import tool
from tavily import TavilyClient

OMDB_API_KEY = os.environ.get("OMDB_API_KEY")
TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

def omdb_api_request(params: dict) -> dict:
    params["apikey"] = OMDB_API_KEY
    base_url = "http://www.omdbapi.com/"
    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        return {"error": f"Error: {e}"}

MOOD_TO_GENRE = {
    "happy": ["Comedy", "Family", "Musical"],
    "sad": ["Drama", "Romance"],
    "excited": ["Action", "Adventure", "Thriller"],
    "scared": ["Horror", "Thriller"],
    "thoughtful": ["Drama", "Documentary", "Mystery"],
    "adventurous": ["Adventure", "Action", "Fantasy"],
    "romantic": ["Romance", "Drama"],
    "nostalgic": ["Family", "Animation", "Drama"]
}

@tool
def findmovies(query: str, year: int = None) -> dict:
    """
    Smart movie recommendation tool. Can search by:
    - Genre/theme: "Recommend a psychological thriller" or "best sci-fi movies"
    - Mood: "I'm feeling adventurous" or "I'm sad"
    - Specific search: "Movies like Inception"
    - Year: Optional year filter
    """
    query_lower = query.lower()

    for mood, genres in MOOD_TO_GENRE.items():
        if mood in query_lower:
            return {
                "recommendation_type": "mood_based",
                "mood_detected": mood,
                "suggested_genres": genres,
                "suggestion": f"For your '{mood}' mood, I recommend {', '.join(genres)} movies. Try asking for a specific title or genre!"
            }

    try:
        search_query = f"{query} highly rated popular movies IMDb best"
        if year:
            search_query += f" {year}"

        tavily_response = tavily_client.search(
            query=search_query,
            search_depth="basic",
            max_results=3
        )

        movies_found = []
        for result in tavily_response.get("results", []):
            content = result.get("content", "")
            if len(movies_found) < 5:
                movies_found.append({
                    "recommendation": content[:200] + "...",
                    "source": result.get("title", "")
                })

        if movies_found:
            return {
                "recommendation_type": "curated_recommendations",
                "query": query,
                "recommendations": movies_found,
                "suggestion": "These are highly-rated movies in this category. Ask for details about a specific title!"
            }
    except Exception as e:
        print(f"Tavily search failed: {e}")

    params = {"s": query, "type": "movie"}
    if year:
        params["y"] = year
    data = omdb_api_request(params)

    if data.get("Response") == "True" and "Search" in data:
        results = []
        for movie in data["Search"][:5]:
            results.append({
                "title": movie.get("Title"),
                "year": movie.get("Year"),
                "imdb_id": movie.get("imdbID")
            })
        return {
            "recommendation_type": "search_results",
            "movies": results,
            "suggestion": f"Found {len(results)} movies. Would you like details about any of these?"
        }

    return {"error": "No movies found. Try a different search term or ask for a specific movie title."}

@tool
def getmoviedetails(title: str = None, imdb_id: str = None) -> dict:
    """Get comprehensive movie details including plot, cast, ratings, awards, and box office."""
    if not title and not imdb_id:
        return {"error": "Provide movie title or IMDb ID"}

    params = {"plot": "full"}
    if imdb_id:
        params["i"] = imdb_id
    else:
        params["t"] = title

    data = omdb_api_request(params)

    if data.get("Response") == "True":
        return {
            "title": data.get("Title"),
            "year": data.get("Year"),
            "rated": data.get("Rated"),
            "released": data.get("Released"),
            "runtime": data.get("Runtime"),
            "genre": data.get("Genre"),
            "director": data.get("Director"),
            "writer": data.get("Writer"),
            "actors": data.get("Actors"),
            "plot": data.get("Plot"),
            "language": data.get("Language"),
            "country": data.get("Country"),
            "awards": data.get("Awards"),
            "imdb_rating": data.get("imdbRating"),
            "imdb_votes": data.get("imdbVotes"),
            "box_office": data.get("BoxOffice"),
            "metascore": data.get("Metascore"),
            "rotten_tomatoes": next((r["Value"] for r in data.get("Ratings", []) if r["Source"] == "Rotten Tomatoes"), None)
        }
    return {"error": data.get("Error", "Movie not found")}

@tool
def wikipediasearch(query: str) -> str:
    """
    Deep analytical search using Wikipedia. Best for:
    - Comparing movies/directors/themes
    - Understanding film history and context
    - Detailed analysis of cinematography, themes, adaptations
    Example: "Compare Nolan and Villeneuve directing styles" or "Inception film analysis"
    """
    try:
        response = tavily_client.search(
            query=f"{query} site:en.wikipedia.org",
            search_depth="advanced",
            max_results=3
        )
        content = "\n\n".join([
            f"**{r['title']}**\n{r['content']}"
            for r in response.get("results", [])
        ])
        return content if content else "No Wikipedia information found."
    except Exception as e:
        return f"Error: {e}"

@tool
def get_recent_reviews_with_rag(movie_title: str) -> str:
    """
    **ADVANCED RAG TOOL** - Fetches recent reviews and news, synthesizes comprehensive answer.
    This demonstrates RAG by: 1) Retrieving multiple sources 2) Augmenting with LLM synthesis
    Best for: Latest movie reviews, critic consensus, recent news about movies.
    """
    try:
        print(f"🔍 Searching for reviews and news about '{movie_title}'...")

        search_response = tavily_client.search(
            query=f"{movie_title} movie reviews critics consensus latest 2024 2025",
            search_depth="advanced",
            max_results=5
        )

        if not search_response.get("results"):
            return f"No recent reviews or news found for '{movie_title}'."

        print(f"📚 Retrieved {len(search_response['results'])} review articles...")

        all_reviews = []
        for result in search_response["results"]:
            all_reviews.append(f"**{result['title']}**\n{result['content']}\n")

        context = "\n\n".join(all_reviews)
        synthesis_prompt = f"""Based on these recent reviews and news articles about '{movie_title}':

{context}

Provide a comprehensive summary covering:
1. Overall critic consensus (positive/mixed/negative)
2. Key strengths highlighted by reviewers
3. Any criticisms or weaknesses mentioned
4. Notable awards, box office performance, or cultural impact

Keep the summary concise but informative."""

        from langchain_google_genai import ChatGoogleGenerativeAI
        llm = ChatGoogleGenerativeAI(
            model="gemini-2.5-flash",
            google_api_key=GEMINI_API_KEY,
            temperature=0.3
        )
        synthesis = llm.invoke(synthesis_prompt)

        print("✅ RAG synthesis complete!")

        return f"**Recent Reviews & Critic Consensus for '{movie_title}'**\n\n{synthesis.content}"

    except Exception as e:
        return f"Error in RAG processing: {e}"

@tool
def findstreamingplatforms(movie_title: str) -> str:
    """Find where to watch a movie in India (Netflix, Prime Video, Disney+ Hotstar, etc.)"""
    try:
        response = tavily_client.search(
            query=f"{movie_title} streaming India Netflix Amazon Prime Disney Hotstar where to watch",
            search_depth="basic",
            max_results=3
        )
        snippets = "\n".join([f"• {r['title']}: {r['content'][:200]}" for r in response.get("results", [])])
        return snippets if snippets else "No streaming information found."
    except Exception as e:
        return f"Error: {e}"

tools = [findmovies, getmoviedetails, wikipediasearch, get_recent_reviews_with_rag, findstreamingplatforms]
print("=" * 70)
print("🎬 ENHANCED MOVIE RECOMMENDATION AGENT - TOOLS READY")
print("=" * 70)
print(f"✅ Created {len(tools)} tools:\n")
for i, tool in enumerate(tools, 1):
    print(f"   {i}. {tool.name}")
print("\n🎯 Key Features:")
print("   • Smart movie recommendations (genre/mood/curated)")
print("   • Comprehensive movie details (ratings, cast, plot)")
print("   • Wikipedia deep analysis (comparisons, themes)")
print("   • Advanced RAG for reviews (BONUS - Retrieve→Synthesize)")
print("   • Streaming platform finder (India focus)")
print("=" * 70)


🎬 ENHANCED MOVIE RECOMMENDATION AGENT - TOOLS READY
✅ Created 5 tools:

   1. findmovies
   2. getmoviedetails
   3. wikipediasearch
   4. get_recent_reviews_with_rag
   5. findstreamingplatforms

🎯 Key Features:
   • Smart movie recommendations (genre/mood/curated)
   • Comprehensive movie details (ratings, cast, plot)
   • Wikipedia deep analysis (comparisons, themes)
   • Advanced RAG for reviews (BONUS - Retrieve→Synthesize)
   • Streaming platform finder (India focus)


In [4]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                      "langgraph", "langchain", "langchain-google-genai",
                      "tavily-python", "requests"])
print("✅ Libraries installed/verified")


✅ Libraries installed/verified


**Step 4: Agent's Brain**

In [5]:
# Build the agent graph – core logic
import operator
from typing import TypedDict, Annotated, Sequence
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import BaseMessage, ToolMessage, HumanMessage
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

print("Building agent logic...")

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.environ.get("GEMINI_API_KEY"),
    temperature=0
)
llm_with_tools = llm.bind_tools(tools)
tool_node = ToolNode(tools)

def call_model(state: AgentState):
    """Agent node - calls the LLM with tools"""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def should_continue(state: AgentState) -> str:
    """Conditional edge - determines if tools should be called"""
    last_message = state["messages"][-1]
    if getattr(last_message, 'tool_calls', None):
        return "continue"
    return "end"

workflow = StateGraph(AgentState)
workflow.add_node("agent", call_model)
workflow.add_node("action", tool_node)
workflow.set_entry_point("agent")
workflow.add_conditional_edges(
    "agent",
    should_continue,
    {"continue": "action", "end": END}
)
workflow.add_edge("action", "agent")

app = workflow.compile()
print("--- LangGraph Agent Compiled Successfully! ---")


Building agent logic...
--- LangGraph Agent Compiled Successfully! ---


** STEP 5: Test the Movie Recommendation Agent**

In [7]:
# Test Agent with Smart Output Formatting
from langchain_core.messages import HumanMessage, ToolMessage
from IPython.display import display, Markdown
import json

def format_movie_data(data):
    """Format movie details into readable markdown"""
    if "title" in data:
        markdown = f"""
## 🎬 {data.get('title', 'N/A')} ({data.get('year', 'N/A')})

### Key Information
- **Rated:** {data.get('rated', 'N/A')}
- **Released:** {data.get('released', 'N/A')}
- **Runtime:** {data.get('runtime', 'N/A')}
- **Genre:** {data.get('genre', 'N/A')}
- **Director:** {data.get('director', 'N/A')}
- **Writers:** {data.get('writer', 'N/A')}
- **Cast:** {data.get('actors', 'N/A')}

### Plot Summary
{data.get('plot', 'No plot available')}

### Critical Reception
- **IMDb Rating:** {data.get('imdb_rating', 'N/A')}/10 ({data.get('imdb_votes', 'N/A')} votes)
- **Metascore:** {data.get('metascore', 'N/A')}
- **Rotten Tomatoes:** {data.get('rotten_tomatoes', 'N/A')}
- **Awards:** {data.get('awards', 'No awards information')}
- **Box Office:** {data.get('box_office', 'N/A')}
"""
        return markdown

    elif "recommendations" in data:
        markdown = "### 🎯 Recommended Movies\n"
        for rec in data.get('recommendations', []):
            markdown += f"- {rec}\n"
        return markdown

    elif "movies" in data:
        markdown = "### 🎬 Movies Found\n"
        for movie in data.get('movies', []):
            markdown += f"- **{movie.get('title', 'N/A')}** ({movie.get('year', 'N/A')})\n"
        return markdown

    return str(data)

# Define your query
query = "Tell me more about Dune Part Two - give me the full plot, cast, ratings, recent reviews, and where to watch it in India"
inputs = {"messages": [HumanMessage(content=query)]}

print("🎬" + "=" * 78 + "🎬")
print(f"  QUERY: {query}")
print("🎬" + "=" * 78 + "🎬\n")

tool_calls_made = []

for output in app.stream(inputs, stream_mode="values"):
    last_message = output["messages"][-1]

    if getattr(last_message, "tool_calls", None):
        print("🔧 AGENT DECISION: Calling Tools")
        print("─" * 80)
        for tool_call in last_message.tool_calls:
            print(f"   🛠️  Tool: {tool_call['name']}")
            print(f"   📋 Arguments: {tool_call['args']}")
            tool_calls_made.append(tool_call['name'])
        print()

    elif isinstance(last_message, ToolMessage):
        print("✅ TOOL EXECUTED SUCCESSFULLY")
        print("─" * 80)

        tool_content = last_message.content

        if isinstance(tool_content, str):
            try:
                tool_data = json.loads(tool_content)
                formatted = format_movie_data(tool_data)
                display(Markdown(formatted))
            except:
                preview = tool_content[:200]
                print(f"{preview}...\n")
        print()

    elif last_message.content and not getattr(last_message, "tool_calls", None):
        print("\n🎯" + "=" * 78 + "🎯")
        print("  FINAL ANSWER")
        print("🎯" + "=" * 78 + "🎯\n")

        answer_text = last_message.content

        if isinstance(answer_text, list) and len(answer_text) > 0:
            answer_text = answer_text[0].get('text', str(answer_text))

        display(Markdown(answer_text))

        print("\n📊" + "=" * 78 + "📊")
        print("  EXECUTION SUMMARY")
        print("📊" + "=" * 78 + "📊")
        print(f"   • Tools Called: {len(tool_calls_made)}")
        if tool_calls_made:
            print(f"   • Tools Used: {', '.join(set(tool_calls_made))}")
        print(f"   • Response Length: {len(str(answer_text))} characters")
        print("   • Status: ✅ Complete and Formatted")
        print("📊" + "=" * 78 + "📊\n")

print("✨ Agent execution complete!")


🎬==============================================================================🎬
  QUERY: Tell me more about Dune Part Two - give me the full plot, cast, ratings, recent reviews, and where to watch it in India
🎬==============================================================================🎬


🎯==============================================================================🎯
  FINAL ANSWER
🎯==============================================================================🎯



Tell me more about Dune Part Two - give me the full plot, cast, ratings, recent reviews, and where to watch it in India


📊==============================================================================📊
  EXECUTION SUMMARY
📊==============================================================================📊
   • Tools Called: 0
   • Response Length: 119 characters
   • Status: ✅ Complete and Formatted
📊==============================================================================📊

🔧 AGENT DECISION: Calling Tools
────────────────────────────────────────────────────────────────────────────────
   🛠️  Tool: getmoviedetails
   📋 Arguments: {'title': 'Dune Part Two'}

✅ TOOL EXECUTED SUCCESSFULLY
────────────────────────────────────────────────────────────────────────────────



## 🎬 Dune: Part Two (2024)

### Key Information
- **Rated:** PG-13
- **Released:** 01 Mar 2024
- **Runtime:** 166 min
- **Genre:** Action, Adventure, Drama
- **Director:** Denis Villeneuve
- **Writers:** Denis Villeneuve, Jon Spaihts, Frank Herbert
- **Cast:** Timothée Chalamet, Zendaya, Rebecca Ferguson

### Plot Summary
Duke Paul Atreides joins the Fremen and begins a spiritual and martial journey to become Muad'dib, while trying to prevent the horrible but inevitable future he's witnessed: a Holy War in his name, spreading throughout the known universe.

### Critical Reception
- **IMDb Rating:** 8.4/10 (713,701 votes)
- **Metascore:** 79
- **Rotten Tomatoes:** 92%
- **Awards:** Won 2 Oscars. 122 wins & 374 nominations total
- **Box Office:** $282,144,358



🔧 AGENT DECISION: Calling Tools
────────────────────────────────────────────────────────────────────────────────
   🛠️  Tool: get_recent_reviews_with_rag
   📋 Arguments: {'movie_title': 'Dune Part Two'}
   🛠️  Tool: findstreamingplatforms
   📋 Arguments: {'movie_title': 'Dune Part Two'}

🔍 Searching for reviews and news about 'Dune Part Two'...
📚 Retrieved 5 review articles...
✅ RAG synthesis complete!
✅ TOOL EXECUTED SUCCESSFULLY
────────────────────────────────────────────────────────────────────────────────
• Dune: Part Two streaming: where to watch online? - JustWatch: Currently you are able to watch "Dune: Part Two" streaming on HBO Max, HBO Max Amazon Channel, YouTube TV. It is also possible to buy "D...



🎯==============================================================================🎯
  FINAL ANSWER
🎯==============================================================================🎯



Here's a comprehensive look at "Dune: Part Two":

**Plot:**
Duke Paul Atreides joins the Fremen and begins a spiritual and martial journey to become Muad'dib, while trying to prevent the horrible but inevitable future he's witnessed: a Holy War in his name, spreading throughout the known universe.

**Cast:**
The film stars Timothée Chalamet, Zendaya, and Rebecca Ferguson.

**Ratings:**
*   **IMDb:** 8.4/10 from 713,701 votes
*   **Rotten Tomatoes:** 92%
*   **Metascore:** 79/100
*   **Awards:** Won 2 Oscars, with a total of 122 wins and 374 nominations.

**Recent Reviews & Critic Consensus:**
"Dune: Part Two" has received overwhelmingly positive reviews, with many critics hailing it as a "modern masterpiece" and an "exceptional sci-fi spectacle." It's frequently cited as one of the greatest sequels ever made.

**Key Strengths:**
*   **Visuals & Spectacle:** Universally praised for its breathtaking cinematography, expansive world-building, and impressive shots, making it a must-see on the biggest screen possible.
*   **Storytelling & Direction:** Denis Villeneuve's direction is lauded for its masterful storytelling, pacing, and ability to explore themes of power and ethnic genocide. The film moves at a brisker pace than its predecessor.
*   **Action:** Features significantly more sci-fi action, including desert guerilla warfare, tightly choreographed fights, sandworm riding, and an explosive third-act battle.
*   **Sound & Score:** The sound design is described as sensational and piercing, enhancing emotional journeys. Hans Zimmer's score is noted as improved, with a memorable love theme.

**Minor Criticisms:**
*   **Pacing & Length:** Despite a brisker pace, some found the film still exposition-heavy, making its 2-hour and 46-minute runtime feel long at times.
*   **CGI:** While improved, some critics felt the CGI wasn't always 100% convincing.
*   **Source Material Divergence:** Fans of the book might find some divergences from the original story.

**Where to Watch in India:**
You can currently rent "Dune: Part Two" on **Prime Video for just Rs. 149**. While it's streaming on HBO Max in some regions, HBO Max is not directly available in India.


📊==============================================================================📊
  EXECUTION SUMMARY
📊==============================================================================📊
   • Tools Called: 3
   • Tools Used: findstreamingplatforms, getmoviedetails, get_recent_reviews_with_rag
   • Response Length: 2182 characters
   • Status: ✅ Complete and Formatted
📊==============================================================================📊

✨ Agent execution complete!
